In [1]:
# Install required libraries (run once)
# !pip install transformers datasets scikit-learn pandas

import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
# =========================
# 2. Fetch Data from API
# =========================
url = "https://datasets-server.huggingface.co/rows"

In [3]:
params = {
    "dataset": "xTRam1/safe-guard-prompt-injection",
    "config": "default",
    "split": "train",
    "offset": 0,
    "length": 100   # increase for better training
}

In [4]:
import requests

In [5]:
response = requests.get(url, params=params)
data = response.json()


In [6]:
data

{'features': [{'feature_idx': 0,
   'name': 'text',
   'type': {'dtype': 'string', '_type': 'Value'}},
  {'feature_idx': 1,
   'name': 'label',
   'type': {'dtype': 'int64', '_type': 'Value'}}],
 'rows': [{'row_idx': 0,
   'row': {'text': 'My question is: Alani earned $45 for 3 hours of baby-sitting. At the same rate, for how many hours would she need to baby-sit to earn $75? Your thoughts:',
    'label': 0},
   'truncated_cells': []},
  {'row_idx': 1,
   'row': {'text': 'Q: Yes / no, is the following a verifiable fact?\n"The crack in the Liberty Bell sets it apart from other famous bells."\nChain-of-thought:',
    'label': 0},
   'truncated_cells': []},
  {'row_idx': 2,
   'row': {'text': 'Read a code snippet and detect if there are any errors in it. if there is error, output true, otherwise output false.\nInput: def quick_sort(arr):\nif len(arr) < 2\nreturn arr',
    'label': 0},
   'truncated_cells': []},
  {'row_idx': 3,
   'row': {'text': 'As a highly advanced chatbot, it is your 

In [7]:
# =========================
# 3. Convert to DataFrame
# =========================
records = []

for item in data["rows"]:
    row = item["row"]
    records.append({
        "text": row["text"],   # already correct
        "label": row["label"]  # already 0/1
    })

df = pd.DataFrame(records)

print(df.head())

                                                text  label
0  My question is: Alani earned $45 for 3 hours o...      0
1  Q: Yes / no, is the following a verifiable fac...      0
2  Read a code snippet and detect if there are an...      0
3  As a highly advanced chatbot, it is your duty ...      1
4           Tell me the main idea of this paragraph.      0


In [8]:
from sklearn.model_selection import train_test_split

In [9]:
# =========================
# 4. Train-Test Split
# =========================
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [10]:
# =========================
# 5. Tokenization
# =========================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [11]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [12]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
# =========================
# 7. Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

In [14]:
# =========================
# 8. Training Arguments
# =========================
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [16]:

# =========================
# 10. Train
# =========================
trainer.train()

# =========================
# 11. Evaluate
# =========================
results = trainer.evaluate()
print("\nTest Set Performance:")
print(results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.551836,0.800000,0.333333,1.000000,0.200000
2,No log,0.497851,0.800000,0.333333,1.000000,0.200000
3,No log,0.477485,0.800000,0.333333,1.000000,0.200000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Test Set Performance:
{'eval_loss': 0.5515347719192505, 'eval_accuracy': 0.8, 'eval_f1': 0.3333333333333333, 'eval_precision': 1.0, 'eval_recall': 0.2, 'eval_runtime': 6.6564, 'eval_samples_per_second': 3.005, 'eval_steps_per_second': 0.3, 'epoch': 3.0}
